# Introduction to Quantum Computing
<img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-SPII-QuantumWorkshop/main/images/microsoft-quantum.png"
     alt="MICROSOFT-QUANTUM"
     width="25%">
<p>(Microsoft Azure, n.d.)

### Prof. David Singletary
### Florida State College at Jacksonville

# Day 4: Oracles, Interference, Noise, and Errors
- Review Day 3 Homework
- Quantum Oracles (Ch. 5)
- Interference
- Errors in Quantum Systems


## Day 3 Homework Solution

- Implement a Binomial Distribution
  - Model number of successes in n trials
  - Apply RY(θ) to each qubit
- Result:
  - amplitudes depend on number of 1s in outcome




In [ ]:
# ---- setup for clone/repo imports ----
import os
import sys
import subprocess
import importlib

REPO_URL = "https://github.com/learnqc/code.git"
REPO_DIR = "/content/code"
SRC_DIR = f"{REPO_DIR}/src"

# Install required pip package(s)
subprocess.run(
    ["pip", "install", "-q", "sty"],
    check=True
)

# Clone repo if needed
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Make src importable
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Clear any stale imports
importlib.invalidate_caches()

print("Setup complete")

In [ ]:
# create a non-uniform distribution across all 3 qubits using rotation gates
# 1. initialize a 3-qubit register and circuit
# 2. choose an angle θ for the RY rotation
# 3. loop over all qubits and apply RY(θ) to each one
# 4. run the circuit to produce the final state
# 5. print the state table to view amplitudes
# 6. measure the state using multiple shots
# 7. print the measurement counts
# 8. change θ and repeat to compare results

from math import pi
from ch03.util import *          # for print_state_table
from ch03.sim_core import *
from ch04.sim_circuit import *

q = QuantumRegister(3)
qc = QuantumCircuit(q)

for i in range(len(q)):
    qc.ry(pi/3, q[i])

state = qc.run()
print_state_table(state)
samples = measure(state, 1000)
print(samples)


In [ ]:
# Add this and change to pi/2; what do you observe?

# group measurement results by number of 1s (Hamming weight)
def group_by_ones(counts, n):
    grouped = {k: 0 for k in range(n + 1)}
    for outcome, count in counts.items():
        bitstring = format(outcome, f"0{n}b")
        ones = bitstring.count('1')
        grouped[ones] += count
    return grouped

q = QuantumRegister(3)
qc = QuantumCircuit(q)

for i in range(len(q)):
    qc.ry(pi/2, q[i])

state = qc.run()
print_state_table(state)
samples = measure(state, 1000)
print(samples)
grouped = group_by_ones(samples, 3)
print(grouped)

### Notes:
- When you use θ = π/2, each qubit behaves like a fair coin, so the raw measurement results (not grouped) show each bitstring with roughly equal frequency, making the distribution look uniform.
- When you group outcomes by the number of 1s, you combine states with the same number of successes, revealing a binomial pattern where middle values occur more often than extremes (there are more ways to form them, i.e., more combinations).
- If you change θ away from π/2, the qubits become biased, and this grouping still reveals a binomial distribution, but now it is skewed instead of symmetric.
- Changing θ biases each qubit, turning a symmetric binomial into a skewed distribution with a tail.
  - The tail can be reversed by increasing the angle past π/2, e.g., 2*π/3.


## Quantum Oracles

- In quantum computation, an oracle is used to recognize specific outcomes and mark them during a computation.
  - We call the outcomes we want to identify “good” outcomes, while all others are “bad” outcomes.
- Oracles typically mark good outcomes in one of two ways:
  - **Phase oracle:** Rotates the amplitudes of the good outcomes by 180°, effectively flipping their phase.
  - **Bit oracle:** Entangles outcomes with an additional qubit, so good outcomes correspond to measuring 1 and bad outcomes correspond to measuring 0.
- These actions mark the states we care about without directly measuring them (because measurement would collapse the quantum state and destroy the superposition).
- The internal implementation of an oracle is independent of how it is later used, which is why it is called an “oracle.”
- Oracles are similar to a SQL WHERE clause
  - They identify which items satisfy a condition
- In classical databases:
  - Writing the condition is easy
  - Searching large datasets can be slow (e.g., full table scan)
- In quantum computing:
  - Defining the oracle (the condition) is hard
  - The search itself is easy using measurement

### Phase Oracles
- A phase oracle marks “good” outcomes by rotating their amplitudes 180° (multiplying by -1), leaving others unchanged.
- Good outcomes are defined using a rule or function (e.g., select outcome 3 from all possibilities).
- When applied to a uniform superposition, only the good outcome's amplitude flips sign, changing its phase but not its magnitude.

### ch05.sim_circuit Shown Here for Review

```
from ch04.sim_core import *
from ch03.sim_gates import *

from ch04.sim_circuit import QuantumRegister as QuantumRegister4
from ch04.sim_circuit import QuantumTransformation as QuantumTransformation4
from ch04.sim_circuit import QuantumCircuit as QuantumCircuit4

import numpy as np

class QuantumRegister(QuantumRegister4):
    pass

class QuantumTransformation(QuantumTransformation4):
    pass

class QuantumCircuit(QuantumCircuit4):

    def initialize(self, state):
        self.state = state

    # this method is covered later in this notebook
    def append(self, circuit, reg):
        assert(reg.size == sum(circuit.regs))
        for tr in circuit.transformations:
            self.transformations.append(QuantumTransformation(
                tr.gate, reg.shift + tr.target, tr.controls, tr.name, tr.arg))

    # this method is covered later in this notebook
    def c_append(self, circuit, c, reg):
        assert(c not in range(reg.shift, reg.shift + reg.size))
        for tr in circuit.transformations:
            self.transformations.append(QuantumTransformation(
                tr.gate, reg.shift + tr.target,
                [c] + [reg.shift + t for t in tr.controls],
                tr.name, tr.arg))
 ```

### Phase Oracle Example
- Start with a 3-qubit system in uniform superposition (8 total states)
- The phase oracle is applied directly to the state
  - (does not change which states exist or their magnitudes)
- The oracle marks the good outcome (k = 3) by flipping the sign of its amplitude
  - Bad outcomes remain unchanged
- The change is not visible in measurement probabilities, but is essential for interference
- The marked state is now distinguishable by phase and can be amplified in later steps

In [ ]:
from ch03.util import *
from ch05.sim_circuit import *

# Textbook function: simulate oracle that takes a state and multiplies
# amplitudes of good outcomes by –1:
def classical_phase_oracle(state, predicate):
  for item in range(len(state)):
    if predicate(item):
      state[item] *= -1

# predicate function: returns True if k == 3
def is_3(k):
    return k == 3

predicate = is_3

# create a 3-qubit register and circuit
q = QuantumRegister(3)
qc = QuantumCircuit(q)

n = len(q)

# initialize to a uniform superposition
qc.initialize([1 / sqrt(2**n) for _ in range(2**n)])

print(f"\nGood outcomes: {[k for k in range(2**n) if predicate(k)]}")

print("Before oracle")
state = qc.run()
print_state_table(state)

# apply the classical phase oracle directly to the state
classical_phase_oracle(state, predicate)

print("After oracle")
print_state_table(state)

### Textbook Exercise 5.1
- Start with n = 4 qubits in a uniform superposition
  - state = [1/sqrt(2**n) for _ in range(2**n)]
- Define a predicate function for good outcomes 2 and 9
- Create a classical phase oracle, apply it to the state, and check that the amplitudes of the good outcomes are rotated by 180°

### Solution
- Start with 4-qubit system in uniform superposition (16 total states)
- Apply the phase oracle
  - Mark good outcomes (k = 2 and k = 9) by flipping their amplitudes

In [ ]:
from ch03.util import *
from ch05.sim_circuit import *

# classical phase oracle (acts directly on the state)
def classical_phase_oracle(state, predicate):
    for item in range(len(state)):
        if predicate(item):
            state[item] *= -1

# predicate function: returns True if k == 2 or 9
def is_2or9(k):
    return k in [2, 9]

predicate = is_2or9

# create a 4-qubit register and circuit
q = QuantumRegister(4)
qc = QuantumCircuit(q)

n = len(q)

# initialize to a uniform superposition
qc.initialize([1/sqrt(2**n) for _ in range(2**n)])

print(f"\nGood outcomes: {[k for k in range(2**n) if predicate(k)]}")

print("Before oracle")
state = qc.run()
print_state_table(state)

# apply the classical phase oracle directly to the state
classical_phase_oracle(state, predicate)

print("After oracle")
print_state_table(state)

### Bit Oracles
- A bit oracle marks good outcomes by entangling them with an additional qubit called a **tag bit**.
- Good outcomes are associated with the tag bit being 1, while others correspond to 0.
- The tag bit can then be used to identify or extract the good outcomes.




### Bit Oracle Example
- Start with a 4-qubit system: 3 data qubits + 1 tag qubit (16 amplitudes total)
  - the data qubits are initialized in a uniform superposition  
  - the tag qubit starts in |0⟩  
  - the oracle is applied directly to the state by redistributing amplitudes
- The oracle moves amplitudes for the “good” outcome (k = 3 → 011)
  - from the portion of the state where the tag bit = 0  
  - to the portion where the tag bit = 1  
- Result:
  - Bad outcomes remain associated with tag bit = 0  
  - The good outcome is now associated with tag bit = 1  
  - This effectively “labels” the desired outcome without changing total probability
- In the 4-bit output table:
  - The good state (011) appears as **1011** (tag bit = 1)  
  - Its original position (0011) is now zeroed out

In [ ]:
from ch03.util import *
from ch05.sim_circuit import *

# apply a bit oracle directly to the state vector
# by moving amplitudes for "good" outcomes into
# the half of the state where the tag bit is 1
def classical_bit_oracle(state, predicate):
    N = len(state)
    state = state + [0 for _ in range(N)]
    for item in range(N):
        if predicate(item):
            state[N + item] = state[item]
            state[item] = 0
    return state

# predicate function: returns True if k == 3
def is_3(k):
    return k == 3

predicate = is_3

# create data register and tag register
q = QuantumRegister(3)
tag = QuantumRegister(1)
qc = QuantumCircuit(q, tag)

n = len(q)

# initialize data qubits in uniform superposition
# and tag qubit in |0>
qc.initialize(
    [1 / sqrt(2**n) for _ in range(2**n)] +
    [0j for _ in range(2**n)]
)

print(f"\nGood outcomes: {[k for k in range(2**n) if predicate(k)]}")

print("Before oracle")
state = qc.run()
print_state_table(state)

# apply the classical bit oracle directly
state = classical_bit_oracle(state, predicate)

print("After oracle")
print_state_table(state)

### Textbook Exercise 5.2
- Create a random state with n = 4 qubits, and apply a bit oracle for good outcome 11.

In [ ]:
from ch03.util import *
from ch05.sim_circuit import *
from math import sqrt
import random

# generate a random normalized quantum state for n qubits
def generate_state(n, seed=None):
    if seed is not None:
        random.seed(seed)

    N = 2**n

    # create random complex amplitudes
    state = [
        complex(random.uniform(-1, 1), random.uniform(-1, 1))
        for _ in range(N)
    ]

    # normalize the state
    norm = sqrt(sum(abs(a)**2 for a in state))
    state = [a / norm for a in state]

    return state

# classical bit oracle (standalone function)
def classical_bit_oracle(state, predicate):
    N = len(state)
    state = state + [0j for _ in range(N)]
    for item in range(N):
        if predicate(item):
            state[N + item] = state[item]
            state[item] = 0j
    return state

# apply a bit oracle to a random 4-qubit state
# to tag the good outcome (k == 11)

# predicate: returns True only for k == 11
def is_11(k):
    return k == 11

predicate = is_11

# create a 4-qubit register and circuit
q = QuantumRegister(4)
qc = QuantumCircuit(q)

n = len(q)

# initialize to a random state
qc.initialize(generate_state(n, seed=42))

print(f"\nGood outcomes: {[k for k in range(2**n) if predicate(k)]}")

print("Before oracle")
state = qc.run()
print_state_table(state)

# apply the classical bit oracle directly to the state
state = classical_bit_oracle(state, predicate)

print("After oracle")
print_state_table(state)

## Implementing Oracles as Circuit Building Blocks
- As circuits grow, it becomes impractical to define everything gate-by-gate in one place  
- Instead, we want to construct circuits by combining smaller, reusable pieces
- This is similar to:
  - Functions in programming  
  - Modules in software design  
- This reduces duplication and simplifies complex designs
- In quantum computing, this is done by composing circuits together
### Circuit Composition with **append**
- The append method is a tool for assembling larger circuits from existing ones
- It allows you to take a fully defined circuit and embed it into another circuit
- Think of it as:
  - “Plug this circuit into that register”
### What Happens Under the Hood
- The append method verifies both circuits use the same number of qubits (required so each gate maps cleanly to corresponding qubit positions in the target register)
  - It iterates through each transformation (gate) in the smaller circuit, then reapplies those transformations onto the new circuit’s register
- Targets are shifted to align with the destination register
- Controls and parameters remain unchanged
### Oracles as Building Blocks
- A circuit that implements an oracle can be built once, then inserted into different algorithms (e.g., search circuits)
- This allows the oracle logic to remain separate from the overall workflow


### Building a Reusable Subcircuit Using Append: Textbook Example
- NOTE: This example isn't using an oracle — it is showing how we'll insert an oracle later.

In [ ]:
# Build a reusable subcircuit that puts qubits in uniform superposition

# Create a 3-qubit circuit, apply X to qubit 0, and inspect state
# Append the subcircuit to the same register and observe final state

from ch03.util import *
from ch05.sim_circuit import *

# build a reusable circuit that creates a uniform distribution
def uniform(n):
    q = QuantumRegister(n)
    qc = QuantumCircuit(q)
    for i in range(len(q)):
        qc.h(q[i])
    return qc

# create a 3-qubit register and main circuit
n = 3
q = QuantumRegister(n)
qc = QuantumCircuit(q)

# apply an X gate to qubit 0
qc.x(q[0])

print("Before appending uniform circuit")
state = qc.run()
print_state_table(state)

# create the uniform circuit
uniform_qc = uniform(n)

# append the uniform circuit onto the existing circuit
qc.append(uniform_qc, q)

print("\nAfter appending uniform circuit")
state = qc.run()
print_state_table(state)

### Additional Composition with **c_append**
- In addition to appending, we can also apply a circuit conditionally
- The c_append method allows us to append a circuit only when a control qubit is 1
- The c_append method parameters are
  - A circuit to append  
  - A control qubit c  
  - A target register reg  
- For each transformation (gate) in the circuit, it
  - applies the same gate to the target register  
  - adds the control qubit c to the gate’s control list  
- If the original gate already had controls:
  - The new control is added to the existing ones
- The control qubit must not be part of the target register
  - This ensures the control condition is independent of the operation
- Conditional controls enables conditional logic in quantum circuits  
- It also supports building more complex behaviors from simple components,  especially important for oracles, controlled subroutines and multi-step algorithms.
- It allows entire subcircuits to behave like controlled gates

### Building a Reusable Subcircuit Using c_append: Example

In [ ]:
# Build a reusable subcircuit for conditional application (H on two qubits)
# Create a 3-qubit circuit, set q[2] = 1, and inspect the state
# Apply the subcircuit to q[0], q[1] only when q[2] = 1

from ch03.util import *
from ch05.sim_circuit import *

def uniform(n):
    q = QuantumRegister(n)
    qc = QuantumCircuit(q)
    for i in range(len(q)):
        qc.h(q[i])
    return qc

# Create a 3-qubit circuit: 2 data qubits + 1 control qubit
q = QuantumRegister(3)
qc = QuantumCircuit(q)

# Set control qubit (q[2]) to 1 so the condition is active
qc.x(q[2])

print("Before c_append")
state = qc.run()
print_state_table(state)

# Create a 2-qubit uniform subcircuit (for q[0], q[1])
uniform_qc = uniform(2)

# Apply the subcircuit ONLY when control qubit q[2] = 1
qc.c_append(uniform_qc, q[2], QuantumRegister(2))

print("\nAfter c_append (controlled uniform)")
state = qc.run()
print_state_table(state)

### Textbook Sidebar: Unitary Transformations
- A unitary transformation is a mathematical operation that transforms a quantum state while preserving total probability
  - It is represented by a matrix of size 2ⁿ x 2ⁿ for n qubits  
  - The sum of all probabilities (|amplitude|²) remains 1  
  - No probability is lost or created
  - The transformation is reversible
- Think of it as rearranging and mixing amplitudes without changing the total probability
- A 1 qubit **unitary** (singular form) acts on pairs of amplitudes  
  - An n-qubit unitary acts on groups of 2ⁿ amplitudes  
- **NOTE:** all valid quantum operations (gates and circuits) must be unitary in order to guarantee physically valid quantum behavior and reversibility of quantum computations.

- e.g., the Hadamard gate (H) is a unitary.

\begin{aligned}
H = \frac{1}{\sqrt{2}}
\begin{bmatrix}
1 & 1 \\
1 & -1
\end{bmatrix}
\end{aligned}

- It turns \(|0⟩ →\) equal superposition, but preserves total probability  
  - Even though the amplitudes change, total probability remains 1





## Building Phase Oracles as Circuits
- We know that a phase oracle flips the sign of amplitudes for “good” outcomes  
- Now consider how to build that behavior as a quantum circuit.
- As an example: Transform → Apply → Undo
  - Convert a target outcome into an all-1s condition
  - Apply a multi-controlled phase gate
  - Undo the conversion
- Details:
  - Use X gates to flip bits where the target has 0
  - Apply mcp(π) so the phase flips only when all controls = 1
    - (mcp = multi-controlled phase gate, applies phase shift when all controls are 1)
  - Reverse the X gates to restore the original basis
- This turns a logical condition into a physical circuit which works for single or multiple outcomes and lets us build reusable oracle circuits to  append them into larger algorithms

  <img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-SPII-QuantumWorkshop/main/images/figure5.9.png"
      alt="Phase Oracle Example"
      width="75%">

- Left side of P(π)
  - X gates applied to convert the target pattern to all 1s
- Middle (P(π))
  - Phase flip happens
- Right side of P(π)
    - X gates appear again on the same qubits (the undo operations)

- Think of it as three independent mini-oracles stitched together:
  - First block marks 001
  - Second block marks 011
  - Third block marks 101
- **NOTE:**
- In the section of the diagram around the final P(π) (marking 101), the X gates simultaneously undo the previous pattern and prepare the next one  
  - Instead of fully resetting between outcomes, the circuit applies only the minimal X changes needed, effectively transitioning directly between patterns (in this example 011 -> 101)


In [ ]:
# Build and apply a phase oracle circuit that flips selected outcomes

from ch03.util import *
from ch05.sim_circuit import *
from math import pi, sqrt

# helper: check if bit k is NOT set in integer m
def is_bit_not_set(m, k):
    return not (m & (1 << k))

# build a phase oracle circuit for given good outcomes
def phase_oracle_match(n, items):
    q = QuantumRegister(n)
    qc = QuantumCircuit(q)

    for m in items:
        # transform: flip bits where m has 0 so target becomes all 1s
        for i in range(n):
            if is_bit_not_set(m, i):
                qc.x(q[i])

        # apply: multi-controlled phase flip (π) on all-1 condition
        qc.mcp(pi, [q[i] for i in range(n - 1)], q[n - 1])

        # undo: restore original basis
        for i in range(n):
            if is_bit_not_set(m, i):
                qc.x(q[i])

    return qc

# example: n = 3 qubits, good outcomes = [1, 3, 5]
n = 3
items = [1, 3, 5]

# build oracle circuit
oracle_circuit = phase_oracle_match(n, items)

# create main circuit and initialize to uniform superposition
q = QuantumRegister(n)
qc = QuantumCircuit(q)

for i in range(n):
    qc.h(q[i])

print(f"\nGood outcomes: {items}")

print("Before oracle")
state = qc.run()
print_state_table(state)

# apply the oracle
qc.append(oracle_circuit, q)

print("After oracle")
state = qc.run()
print_state_table(state)

## Hands-On: Build a Phase Oracle for One Marked Outcome
- Create a reusable phase oracle circuit for a 3-qubit system that marks one chosen outcome by flipping the sign of its amplitude.
  - use X gates to convert a target pattern into an all-1s condition
  - use mcp(pi) to apply a phase flip
  - append the oracle to a larger circuit
  1. Create a helper function is_bit_not_set(m, k)  
    - It should return True when bit k of integer m is 0
  2. Write a function phase_oracle_match(n, items)  
   - It should create a quantum circuit on n qubits
   - For each marked outcome in items:
     - apply X to qubits whose corresponding bit is 0
     - apply mcp(pi) so the phase flips only on the all-1s case
     - undo the X gates
  3. Set:
   - n = 3
   - items = [6]
  4. Create a new 3-qubit circuit and apply H to every qubit  
   - This creates a uniform superposition
  5. Run the circuit and print the state table before applying the oracle
  6. Append the oracle circuit to the main circuit
  7. Run the circuit again and print the state table after the oracle
  8. Compare the two state tables  
   - Which amplitude changed sign?
   - Did any magnitudes change?
### Expected Results
- All 8 outcomes are present before and after the oracle
- Only the marked outcome has its sign flipped
- The magnitudes stay the same
- The oracle changes phase, not probability directly

In [ ]:
# Hands-On Workspace
from ch03.util import *
from ch05.sim_circuit import *
from math import pi

# helper: return True if bit k of m is 0
def is_bit_not_set(m, k):
    return not (m & (1 << k))

# build a phase oracle circuit for selected good outcomes
def phase_oracle_match(n, items):
    q = QuantumRegister(n)
    qc = QuantumCircuit(q)

    for m in items:
        # TODO: remove "pass" and transform target pattern into all 1s
        pass

        # TODO: apply phase flip on the all-1s condition

        # TODO: restore original basis

    return qc

# TODO: set n and mark outcome 6 (110)

# TODO: build the oracle circuit using the phase_oracle_match function

# TODO: create main circuit

# TODO: initialize uniform superposition

# TODO: print the good outcomes

print("Before oracle")
# TODO run the circuit and print the state table

print("After oracle")
# TODO run the circuit again and print the state table

### Hands-On Solution

In [ ]:
# Hands-On Solution
from ch03.util import *
from ch05.sim_circuit import *
from math import pi

# helper: return True if bit k of m is 0
def is_bit_not_set(m, k):
    return not (m & (1 << k))

# build a phase oracle circuit for selected good outcomes
def phase_oracle_match(n, items):
    q = QuantumRegister(n)
    qc = QuantumCircuit(q)

    for m in items:
        # transform target pattern into all 1s
        for i in range(n):
            if is_bit_not_set(m, i):
                qc.x(q[i])

        # apply phase flip on the all-1s condition
        qc.mcp(pi, [q[i] for i in range(n - 1)], q[n - 1])

        # restore original basis
        for i in range(n):
            if is_bit_not_set(m, i):
                qc.x(q[i])

    return qc

# set n and mark outcome 6 (110)
n = 3
items = [6]

# build the oracle circuit using the phase_oracle_match function
oracle_circuit = phase_oracle_match(n, items)

# create main circuit
q = QuantumRegister(n)
qc = QuantumCircuit(q)

# initialize uniform superposition
for i in range(n):
    qc.h(q[i])

# print the good outcomes
print(f"\nGood outcomes: {items}")

print("Before oracle")
# run the circuit and print the state table
state = qc.run()
print_state_table(state)

# append the oracle circuit
qc.append(oracle_circuit, q)

print("After oracle")
# run the circuit again and print the state table
state = qc.run()
print_state_table(state)

### Building Bit Oracles as Circuits
- Recall that a bit oracle marks good outcomes by setting a tag qubit = 1
  - Instead of changing phase, it moves/copies amplitudes into states where the tag bit = 1
- As an example (again, using the same core pattern as phase oracles):
  - Transform → Apply → Undo
    - Convert a target outcome to all 1s
    - Use a multi-controlled X (MCX) to flip the tag qubit
    - Undo the transformation
- Details
  - Use X gates to flip bits where the target has 0
  - Apply mcx(...) so the tag qubit flips only when all controls = 1
    - (multi-controlled X: flips the target qubit when all controls are 1)
  - Reverse the X gates to restore the original basis
- A new qubit (ancilla/tag) is added which starts in |0⟩ and becomes 1 only for good outcomes
- This means that the system grows from n qubits to n+1 qubits and turns a logical condition into a measurable result.
- Unlike phase oracles, you can directly observe the tag bit.
- This enables filtering, post-selection (keeping only results where a chosen condition is met), and hybrid quantum-classical workflows ((combining quantum circuits with classical processing steps)
- How is this different from a phase oracle?
  - A phase oracle flips the sign (phase) of marked outcomes without changing their location, while a bit oracle moves/labels marked outcomes by setting a tag qubit to 1.
    - A phase oracle changes how states interfere later, not what you see immediately  
    - A bit oracle, by contrast, makes the result immediately observable by setting a tag qubit to 1



In [ ]:
# Build and apply a bit oracle circuit that tags selected outcomes

from ch03.util import *
from ch05.sim_circuit import *
from math import sqrt

# helper: check if bit k is NOT set in integer m
def is_bit_not_set(m, k):
    return not (m & (1 << k))

# build a bit oracle circuit for given good outcomes
def bit_oracle_match(n, items):
    q = QuantumRegister(n)
    a = QuantumRegister(1)
    qc = QuantumCircuit(q, a)

    for m in items:
        # transform: flip bits where m has 0 so target becomes all 1s
        for i in range(n):
            if is_bit_not_set(m, i):
                qc.x(q[i])

        # apply: flip tag qubit when all controls = 1
        qc.mcx([q[i] for i in range(n)], a[0])

        # undo: restore original basis
        for i in range(n):
            if is_bit_not_set(m, i):
                qc.x(q[i])

    return qc

# example: n = 3 qubits, good outcomes = [3]
n = 3
items = [3]

# build oracle circuit
oracle_circuit = bit_oracle_match(n, items)

# create main circuit (data + ancilla)
q = QuantumRegister(n)
a = QuantumRegister(1)
qc = QuantumCircuit(q, a)

# initialize data qubits in uniform superposition
for i in range(n):
    qc.h(q[i])

print(f"\nGood outcomes: {items}")

print("Before oracle")
state = qc.run()
print_state_table(state)

# apply the oracle (note: n+1 qubits required)
qc.append(oracle_circuit, QuantumRegister(n + 1))

print("After oracle")
state = qc.run()
print_state_table(state)

## Hands-On: Build a Bit Oracle for One Marked Outcome
- Create a reusable bit oracle circuit for a 3-qubit system that marks one chosen outcome by setting a tag qubit to 1.
  - use X gates to convert a target pattern into an all-1s condition  
  - use mcx to flip the tag qubit when all controls = 1  
  - append the oracle to a larger circuit  
1. Create a helper function `is_bit_not_set(m, k)`  
   - It should return True when bit k of integer m is 0  
2. Write a function bit_oracle_match(n, items)  
   - It should create a quantum circuit on:
     - n data qubits  
     - 1 tag qubit  
   - For each marked outcome in items:
     - apply X to qubits whose corresponding bit is 0  
     - apply mcx(...) so the tag flips only on the all-1s case  
     - undo the X gates  
3. Set:
   - n = 3  
   - items = [6]  
4. Create a new circuit with:
   - 3 data qubits  
   - 1 ancilla qubit  
5. Apply H to each data qubit  
   - This creates a uniform superposition  
6. Run the circuit and print the state table before applying the oracle  
7. Append the oracle circuit to the main circuit  
   - Be sure to pass a register with `n + 1` qubits  
8. Run the circuit again and print the state table after the oracle  
9. Compare the two state tables  
   - Which outcomes now have tag bit = 1?  
   - Did any amplitudes disappear or move?     
### Expected Results
- The oracle separates good outcomes rather than changing phase  
- Before the oracle:
  - All amplitudes are in states where tag bit = 0  
- After the oracle:
  - The marked outcome appears in the lower half of the state table with tag bit = 1
  - Its original position (tag = 0) is zeroed out    
- The oracle changes the location of amplitudes, not their magnitude  

In [ ]:
# Hands-On Workspace

from ch03.util import *
from ch05.sim_circuit import *

# helper: return True if bit k of m is 0
def is_bit_not_set(m, k):
    return not (m & (1 << k))

# build a bit oracle circuit for selected good outcomes
def bit_oracle_match(n, items):
    q = QuantumRegister(n)
    a = QuantumRegister(1)
    qc = QuantumCircuit(q, a)

    for m in items:
        # TODO: remove "pass" and transform target pattern into all 1s
        # using "if is_bit_not_set(...)"
        for i in range(n):
          pass

        # TODO: flip the tag qubit when all controls are 1 using mcx

        # TODO: restore original basis using "if is_bit_not_set(...)"

    return qc

# TODO: set n and mark outcome 6 (110)

# TODO: build the oracle circuit using the bit_oracle_match function

# TODO: create main circuit with data qubits and one tag qubit

# TODO: initialize uniform superposition on the data qubits only

# TODO: print the good outcomes

print("Before oracle")
# TODO: run the circuit
# TODO: print the state table

# TODO: append the oracle circuit to a register with n + 1 qubits

print("After oracle")
# TODO: run the circuit again
# TODO: print the state table

### Hands-On Solution

In [ ]:
# Hands-On Solution

from ch03.util import *
from ch05.sim_circuit import *

# helper: return True if bit k of m is 0
def is_bit_not_set(m, k):
    return not (m & (1 << k))

# build a bit oracle circuit for selected good outcomes
def bit_oracle_match(n, items):
    q = QuantumRegister(n)
    a = QuantumRegister(1)
    qc = QuantumCircuit(q, a)

    for m in items:
        # transform target pattern into all 1s
        # using "if is_bit_not_set(...)"
        for i in range(n):
            if is_bit_not_set(m, i):
                qc.x(q[i])

        # flip the tag qubit when all controls are 1 using mcx
        qc.mcx([q[i] for i in range(n)], a[0])

        # restore original basis using "if is_bit_not_set(...)"
        for i in range(n):
            if is_bit_not_set(m, i):
                qc.x(q[i])

    return qc

# set n and mark outcome 6 (110)
n = 3
items = [6]

# build the oracle circuit using the bit_oracle_match function
oracle_circuit = bit_oracle_match(n, items)

# create main circuit with data qubits and one tag qubit
q = QuantumRegister(n)
a = QuantumRegister(1)
qc = QuantumCircuit(q, a)

# initialize uniform superposition on the data qubits only
for i in range(n):
    qc.h(q[i])

# print the good outcomes
print(f"\nGood outcomes: {items}")

print("Before oracle")
# run the circuit
state = qc.run()
# print the state table
print_state_table(state)

# append the oracle circuit to a register with n + 1 qubits
qc.append(oracle_circuit, QuantumRegister(n + 1))

print("After oracle")
# run the circuit again
state = qc.run()
# print the state table
print_state_table(state)

## Interference
(Microsoft Quantum n.d.)
- Quantum interference comes from the wave-like nature of quantum states.
- It cccurs when a system is in superposition
  - States combine (interfere) with each other
  - Constructive interference increases probability
  - Destructive interference decreases probability
  - Gates create superposition and control phase
- Interference shapes measurement outcomes by amplifying correct answers and suppressing incorrect ones
  - The goal is to make the right answer more likely to be observed
- e.g.,
  - Grover’s Algorithm uses interference to boost the probability of the correct search result
  - Quantum Fourier Transform (QFT) uses interference between states to compute transforms efficiently
  - Quantum Phase Estimation extracts phase information using interference patterns
- Interference is also used in error detection and correction
  - It helps identify when quantum states have been disrupted

- **How Interference is Implemented in Circuits**  
  - Quantum gates create superposition (e.g., Hadamard) and control phase (e.g., Z, phase rotations)  
  - Circuits are designed so amplitudes combine in specific ways  
    - Desired states experience constructive interference (amplified)  
    - Undesired states experience destructive interference (suppressed)  
  - This engineered interference is what drives quantum algorithms like Grover's and QFT
- **Role of Phase**  
	- Interference depends on relative phase differences between quantum states  
	- States with the same magnitude can combine differently depending on phase  
	- Phase determines whether interference is constructive or destructive
- **Interference vs Classical Probability**  
	- Classical systems add probabilities directly  
	- Quantum systems add amplitudes first, then square them  
	- This allows destructive interference, where outcomes can cancel out completely  
- **Interference as Amplitude Redistribution**  
	- Interference does not create or remove probability  
	- It redistributes probability across states  
	- Some outcomes are amplified while others are suppressed  
- **Failure of Interference (Noise Link)**  
	- Noise and decoherence disrupt phase relationships between states  
	- This weakens or destroys interference patterns  
	- Results become more random and less useful  


In [ ]:
# constructive interference example

# first H creates an equal superposition of |0⟩ and |1⟩
# phase oracle flips the sign of the |1⟩ amplitude
# second H recombines the amplitudes
# result:
# |0⟩ cancels out
# |1⟩ is amplified

from ch03.util import *
from ch05.sim_circuit import *

# mark |1> as the desired state
def predicate(k):
    return k == 1

# classical phase oracle (acts directly on the state)
def classical_phase_oracle(state, predicate):
    for item in range(len(state)):
        if predicate(item):
            state[item] *= -1

# create a 1-qubit register and circuit
q = QuantumRegister(1)
qc = QuantumCircuit(q)

# step 1: create equal superposition
qc.h(q[0])
state = qc.run()

print("After first H")
print_state_table(state)

# step 2: mark the desired state by flipping its phase
classical_phase_oracle(state, predicate)
print("After phase oracle")
print_state_table(state)

# step 3: recombine amplitudes
qc.h(q[0])
state = qc.run()
print("After second H")
print_state_table(state)

In [ ]:
# destructive interference example

# undesired state |0⟩ starts with nonzero amplitude
# after phase marking (bit flipping) and recombination,
# contributions to |0⟩ cancel out and amplitude goes to
# (approximately) zero

from ch03.util import *
from ch05.sim_circuit import *

# classical phase oracle (acts directly on the state)
def classical_phase_oracle(state, predicate):
    for item in range(len(state)):
        if predicate(item):
            state[item] *= -1
from math import sqrt

# mark |1> as the desired state
def predicate(k):
    return k == 1

# create a 1-qubit register and circuit
q = QuantumRegister(1)
qc = QuantumCircuit(q)

# step 1: create equal superposition
qc.h(q[0])
state = qc.run()

print("After first H")
print_state_table(state)

# step 2: flip phase of the desired state |1>
classical_phase_oracle(state, predicate)

print("After phase oracle")
print_state_table(state)

# step 3: recombine amplitudes
qc.h(q[0])
state = qc.run()

print("After second H")
print_state_table(state)

# Errors in Quantum Systems
(Microsoft Quantum n.d.)
- Quantum errors arise from several sources that affect fragile quantum states
- One of the primary sources of these errors is noise, which disrupts and destabilizes quantum states


## Fragility of Quantum States
- Quantum states are inherently fragile and highly sensitive to their environment  
- Even small disturbances can disrupt superposition and entanglement  
- This sensitivity leads to more frequent errors compared to classical systems  




## Noise
- Noise in quantum systems refers to environmental and technical disturbances that disrupt fragile quantum states  
- Because qubits must remain isolated, interactions with heat, electromagnetic radiation, material defects, or imperfect control signals introduce noise  
- Noise from the surrounding environment disturbs qubit states and can lead to decoherence, causing qubits to lose their quantum properties over time  
- Understanding and mitigating noise is essential for improving quantum hardware performance and enabling scalable operations  



## Other Error Causes
- Imperfect gate operations can also introduce errors due to limitations in hardware precision  
- These errors can result from inaccuracies in control signals and limitations in how precisely gates are implemented  
- They can lead to incorrect transformations of quantum states, reducing computational reliability  
- Understanding and improving gate precision is important for increasing the accuracy of quantum computations  
## Error Rates: Classical vs Quantum
- Error rates differ significantly between classical and quantum computers, with quantum systems experiencing errors far more frequently  
- Classical computers have extremely low error rates, often measured in errors per billion (EPB) or errors per trillion (EPT) operations  
- Quantum computers have much higher error rates, typically around 1% to 0.1% per operation  
- This means that, on average, approximately one out of every 100 to 1000 quantum gate operations may result in an error  
## Error Mitigation and Error Correction
- Understanding and dealing with noise is essential for improving hardware performance and enabling scalable systems  
- Handling errors is critical for ensuring that quantum algorithms produce reliable and meaningful results  
- “Today's quantum computers rely on **error mitigation (EM)**, but scalable quantum computing will require full ** quantum error correction (QEC)** (distinguished from classical error correction).
- Error mitigation does not fix the errors directly; it reduces the impact of noise and imperfections by adjusting results after computation.
  - Methods include statistical techniques, calibration adjustments, and post-processing corrections
  - These do not eliminate errors entirely but help extract more reliable information from noisy quantum devices.
- Error correction detects and fixes errors during computation by encoding information across multiple physical qubits into more stable logical qubits, helping preserve fragile quantum states despite noise and decoherence.
- It does not eliminate all errors completely; its effectiveness depends on:  
	- The specific error-correcting code used  
	- The underlying hardware noise levels
- Requires a lot of qubits and overhead
- Not yet widely available on current quantum hardware
- Remains an active area of research and development


## Types of Errors
- In classical computing, errors typically occur as bit flips, where a 0 changes to a 1 or a 1 changes to a 0  
- In quantum computing, errors are more complex and can occur in multiple ways:  
	- Bit flips (similar to classical errors)  
	- Phase flips, which alter the phase of a quantum state without changing its measured value directly  
	- Combinations of bit and phase flips, making quantum errors more difficult to detect and correct  


## Error Mitigation Example Using Post-Processing

In [ ]:
!pip install qiskit
!pip install qiskit_aer

In [ ]:
# Example: simple error mitigation with a noisy simulator
# This demonstrates mitigation, not full quantum error correction

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from collections import Counter

# build a simple circuit
qc = QuantumCircuit(1, 1)
qc.h(0)              # create equal superposition
qc.measure(0, 0)

# ideal simulator (no noise)
ideal_sim = AerSimulator()

# noisy simulator
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.08, 1), ["h"]
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.05, 1), ["measure"]
)

noisy_sim = AerSimulator(noise_model=noise_model)

shots = 2000

# run ideal version
ideal_job = ideal_sim.run(transpile(qc, ideal_sim), shots=shots)
ideal_counts = ideal_job.result().get_counts()

# run noisy version
noisy_job = noisy_sim.run(transpile(qc, noisy_sim), shots=shots)
noisy_counts = noisy_job.result().get_counts()

print("Ideal counts: ", ideal_counts)
print("Noisy counts: ", noisy_counts)

# convert counts to probabilities
def counts_to_probs(counts, shots):
    return {
        "0": counts.get("0", 0) / shots,
        "1": counts.get("1", 0) / shots
    }

ideal_probs = counts_to_probs(ideal_counts, shots)
noisy_probs = counts_to_probs(noisy_counts, shots)

print("\nIdeal probabilities: ", ideal_probs)
print("Noisy probabilities: ", noisy_probs)

# simple mitigation idea:
# assume measurement noise shifts probabilities slightly toward the wrong answer
# estimate a correction using a known calibration matrix
#
# Example calibration matrix:
# true 0 measured as 0 with 95% probability
# true 1 measured as 1 with 95% probability
#
# [measured_0]   [0.95 0.05] [true_0]
# [measured_1] = [0.05 0.95] [true_1]

a = 0.95
b = 0.05

m0 = noisy_probs["0"]
m1 = noisy_probs["1"]

# invert the 2x2 calibration matrix
det = a * a - b * b
true0 = (a * m0 - b * m1) / det
true1 = (-b * m0 + a * m1) / det

# clamp values into valid probability range
true0 = max(0, min(1, true0))
true1 = max(0, min(1, true1))

# renormalize
total = true0 + true1
mitigated_probs = {
    "0": true0 / total,
    "1": true1 / total
}

print("\nMitigated probabilities: ", mitigated_probs)

## More on Quantum Error Correction
- Quantum error correction protects information by encoding it across multiple qubits.
- Instead of storing information in a single qubit, it is distributed across several physical qubits, which are the actual hardware qubits and are prone to noise and errors.
- This encoding creates a logical qubit, which represents the protected information spread across those physical qubits. Logical qubits are more stable and form the foundation for reliable quantum computation.
- By distributing information in this way, errors can be detected and corrected without directly measuring and collapsing the quantum state.
## Common Quantum Error Correction Codes
- **Shor code**  
	- Uses 9 physical qubits to encode 1 logical qubit  
	- Can correct both bit flip and phase flip errors  
- **Steane code**  
	- Uses 7 physical qubits  
	- Can correct both bit and phase errors  
	- Designed to be fault-tolerant, meaning the correction process does not introduce additional errors.
- **Surface code**  
	- Uses a 2D lattice of qubits  
	- Has a high error tolerance threshold  
	- Considered one of the most promising approaches for large-scale quantum computing  
- **Hastings-Haah code**  
	- Offers improved efficiency for certain hardware architectures (e.g., Majorana qubits)  
	- Less practical for standard gate-based quantum systems  


## Shor Code Example

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit import transpile

# 9 physical qubits + 9 classical bits
q = QuantumRegister(9)
c = ClassicalRegister(9)
qc = QuantumCircuit(q, c)

# ----------------------------------------
# Step 1: Prepare logical qubit |ψ⟩
# (use |+⟩ state for visibility)
# ----------------------------------------
qc.h(q[0])

# ----------------------------------------
# Step 2: Encode (Shor Code)
# ----------------------------------------

# Phase-flip protection (first layer)
qc.cx(q[0], q[3])
qc.cx(q[0], q[6])

# Convert to bit-flip code using Hadamards
qc.h(q[0])
qc.h(q[3])
qc.h(q[6])

# Bit-flip protection (repeat each qubit 3 times)
for i in [0, 3, 6]:
    qc.cx(q[i], q[i+1])
    qc.cx(q[i], q[i+2])

# ----------------------------------------
# Step 3: Introduce an error
# (simulate noise manually)
# ----------------------------------------

qc.x(q[1])   # bit flip error
qc.z(q[4])   # phase flip error

# ----------------------------------------
# Step 4: Decode (simplified correction)
# ----------------------------------------

# Reverse bit-flip encoding
for i in [0, 3, 6]:
    qc.cx(q[i], q[i+1])
    qc.cx(q[i], q[i+2])

# Undo Hadamards
qc.h(q[0])
qc.h(q[3])
qc.h(q[6])

# Reverse phase encoding
qc.cx(q[0], q[3])
qc.cx(q[0], q[6])

# ----------------------------------------
# Step 5: Measure
# ----------------------------------------
qc.measure(q, c)

# Run simulation
sim = AerSimulator()
compiled = transpile(qc, sim)
result = sim.run(compiled, shots=1024).result()

counts = result.get_counts()
print("Measurement results:", counts)

## References

- Gonciulea, C., & Stefanski, C. (2025). Building Quantum Software With Python: A Developer’s Guide. Manning Publications. ISBN: 978-1633437630.
- Microsoft Azure. (n.d.). Quantum computing solutions. https://azure.microsoft.com/en-gb/solutions/quantum-computing
- Microsoft Quantum. (n.d.). Quantum interference.
https://quantum.microsoft.com/en-us/insights/education/concepts/interference
- Microsoft Quantum. (n.d.). Quantum Error Correction. https://quantum.microsoft.com/en-us/insights/education/concepts/quantum-error-correctio